In [ ]:
import sys
import os
here = os.getcwd()
sys.path.append(os.path.join(here, "Python/"))

from ddl import execute

**Section 29** (Prohibition to engage in credit activities)
<ul style="list-style-type: none;">
<li>(1) A person must not engage in a credit activity if the person does not hold a licence authorising the person to engage in the credit activity.</li>
<li>(3) For the purposes of subsections (1) and (2), it is a defence if:
    <ul style="list-style-type: none;">
    <li>(a) the person engages in the credit activity on behalf of another person (the principal); and</li>
    <li>(b) the person is:
        <ul style="list-style-type: none;">
        <li>(i) an employee or director of the principal or of a related body corporate of the principal; or</li>
        <li>(ii) a credit representative of the principal.</li>
        </ul>
    </li>
    </ul>
</li>
</ul>

In [ ]:
rules = """
r1: => [O]~personEngageCreditActivity
r2: personHoldAuthorizingLicense => [P]personEngageCreditActivity
r2>r1
r3: personOnBehalfOfPrincipal & personWorkForPrincipal => [P]personEngageCreditActivity
r3>r1

r3a1: personEmployeePrincipal => personWorkForPrincipal 
r3a2: personDirectorPrincipal => personWorkForPrincipal
r3a3: personEmployeeBodyCorporatePrincipal => personWorkForPrincipal
r3a4: personDirectorBodyCorporatePrincipal => personWorkForPrincipal

r4: personOnBehalfOfPrincipal & personCreditRepresentativePrincipal => [P]personEngageCreditActivity
r4>r1
"""

facts = """
personDirectorPrincipal
personOnBehalfOfPrincipal
"""

execute(rules+facts)

Alternative encoding using a predicate logic style representation

In [ ]:
rules = """
r1: => [O]~engageCreditActivity(person)
r2: holdAuthorizingLicense(person) => [P]engageCreditActivity(person)
r2>r1
r3: onBehalfOf(person,principal) & workFor(person,principal) => [P]engageCreditActivity(person)
r3>r1
r3a1: employee(person,principal) => workFor(person,principal)
r3a2: director(person,principal) => workFor(person,principal)
r3a3: employee(person,bodyCorporate) & own(principal,bodyCorporate) => workFor(person,principal)
r3a4: director(person,bodyCorporate) & own(principal,bodyCorporate) => workFor(person,principal)

r4: onBehalfOf(person,principal) & creditRepresentative(person,principal) => [P]engageCreditActivity(person)
r4>r1
"""

facts = """
director(person,principal)
creditRepresentative(person,principal)
onBehalfOf(person,principal)
"""

facts2 = """
holdAuthorizingLicense(person)
"""

execute(rules+facts2)

The implementation of Defeasible Deontic Logic in ASP allows us to use to represent norms with variables (and ground them before the reasoning process).  

At this stage, the encoding using variables has a few additional requirements. 

1. all atoms must be declared with `atom(...)`
2. all rules must be grounded before the reasoning process. 

This means that we have to ensure that all ASP clauses are safe. To this end, we can define a domain for the variables and then use to make the ASP encoding safe.

```prolog
atom(predicate(Var1,Var2,...)) :-
    domain1(Var1), domain2(Var2), ...
```

A rule in the ASP encoding is then represented as follows:

```prolog
<type>Rule(label,head).
body(label,(atom1;atom2;...)).
```
Where `<type>Rule` is either `constitutiveRule`, `prescriptiveRule` or `permissiveRule`. When we introduce the variables in the rules we have to the two ASP clauses safe. 
```prolog
<type>Rule(label(Var1,Var2,...),head(Var1,Var2,...)):-
    domain1(Var1), domain2(Var2), ...
body(label(Var1,Var2,...),(atom1(Var1,Var2,...);atom2(Var1,Var2,...);...)) :-
    domain1(Var1), domain2(Var2), ...
```

In [ ]:
domain = """
person(alice;bob;charlie).
corporation(abc;acme).
legalEntity(X) :- person(X).
legalEntity(X) :- corporation(X).
"""

atoms = """
atom(director(Person,Principal)) :- 
    person(Person), corporation(Principal).
atom(employee(Person,Principal)) :- 
    person(Person), legalEntity(Principal), Person!=Principal.
atom(engageCreditActivity(Person)) :- person(Person).
atom(holdAuthorizingLicense(Person)) :- person(Person).
atom(onBehalfOf(Person,Principal)) :- 
    person(Person), legalEntity(Principal), Person!=Principal.
atom(workFor(Person,Principal)) :- 
    person(Person), legalEntity(Principal), Person!=Principal.
atom(creditRepresentative(Person,Principal)) :- 
    person(Person), legalEntity(Principal), Person!=Principal.
atom(own(Principal,BodyCorporate)) :- 
    legalEntity(Principal), corporation(BodyCorporate), Principal!=BodyCorporate.
"""

rules = """
prescriptiveRule(r1(Person),non(engageCreditActivity(Person))) :- 
    person(Person).

permissiveRule(r2(Person),engageCreditActivity(Person)) :- 
    person(Person).
body(r2(Person),holdAuthorizingLicense(Person)) :-
    person(Person).

superior(r2(Person),r1(Person)) :- person(Person).

constitutiveRule(r3a1(Person,Principal),workFor(Person,Principal)) :-
    person(Person), legalEntity(Principal), Person!=Principal.
body(r3a1(Person,Principal),employee(Person,Principal)) :-
    person(Person), legalEntity(Principal), Person!=Principal.

constitutiveRule(r3a2(Person,Principal),workFor(Person,Principal)) :-
    person(Person), corporation(Principal), Person!=Principal.
body(r3a2(Person,Principal),director(Person,Principal)) :-
    person(Person), corporation(Principal), Person!=Principal.

constitutiveRule(r3a3(Person,Principal),workFor(Person,Principal)) :-
    person(Person), legalEntity(Principal), Person!=Principal.
body(r3a3(Person,Principal),
     (employee(Person,BodyCorporate);own(Principal,BodyCorporate))) :-
    person(Person), legalEntity(Principal), corporation(BodyCorporate), Principal != BodyCorporate, Person!=Principal.

constitutiveRule(r3a4(Person,Principal),workFor(Person,Principal)) :-
    person(Person), legalEntity(Principal), Person!=Principal.
body(r3a4(Person,Principal),
     (director(Person,BodyCorporate);own(Principal,BodyCorporate))) :-
    person(Person), legalEntity(Principal), corporation(BodyCorporate),
    Principal!= BodyCorporate, Person!=Principal.

permissiveRule(r3(Person),engageCreditActivity(Person)) :- 
    person(Person).
body(r3(Person),
     (onBehalfOf(Person,Principal);workFor(Person,Principal))) :-
    person(Person), legalEntity(Principal), Person!=Principal.

superior(r3(Person),r1(Person)) :- person(Person).

permissiveRule(r4(Person,Principal),engageCreditActivity(Person)) :-
    person(Person), legalEntity(Principal), Person!=Principal.
body(r4(Person,Principal),
        (onBehalfOf(Person,Principal);creditRepresentative(Person,Principal))) :-
    person(Person), legalEntity(Principal), Person!=Principal.
    
superior(r4(Person,Principal),r1(Person)) :- person(Person), legalEntity(Principal), Person!=Principal.
"""

facts = """
fact(holdAuthorizingLicense(alice)).
fact(onBehalfOf(bob,abc)).
fact(creditRepresentative(bob,abc)).
"""

execute(domain+atoms+rules+facts,"asp")

execute(domain+atoms+rules,"asp")